In [ ]:
!pip install -q librosa tensorflow scikit-learn matplotlib

In [ ]:
import os
import zipfile
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (Conv2D,MaxPooling2D,Flatten,Dense,Dropout)
print("TensorFlow version:", tf.version)

In [ ]:
def choose_file(prompt):
    """Загрузка в Colab или ввод пути в обычном Jupyter."""
    try:
        from google.colab import files
        uploaded = files.upload()
        return next(iter(uploaded))
    except ImportError:
        return input(prompt).strip()

zip_name = choose_file("Введите путь к ZIP-архиву с датасетом: " )

In [ ]:
extract_path = "dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

In [ ]:
audio_files = []
for root, dirs, files_list in os.walk(extract_path):
 for file in files_list:

    if file.lower().endswith(".wav"):

        audio_files.append(
            os.path.join(root, file)
        )
print("Audio files:", len(audio_files))

In [ ]:
def get_label(file_path):

  filename = os.path.basename(file_path)

  first_letter = filename[0].upper()

  if first_letter == "F":
    return "food"

  elif first_letter == "B":
    return "brushing"

  elif first_letter == "I":
    return "isolation"

  else:
    return None


In [ ]:
data = []
for file_path in audio_files:

  label = get_label(file_path)

  if label is not None:

    data.append({
        "file": file_path,
        "label": label
    })
df = pd.DataFrame(data)
print(df.head())
print("\nКоличество записей:")
print(len(df))
print("\nКлассы:")
print(df["label"].value_counts())


In [ ]:
def audio_to_mel(file_path):

 audio, sr = librosa.load(
    file_path,
    sr=16000,
    duration=4
)

 target_length = sr * 4

 if len(audio) < target_length:

    audio = np.pad(
        audio,
        (0, target_length - len(audio))
    )

 else:

    audio = audio[:target_length]

 mel = librosa.feature.melspectrogram(
    y=audio,
    sr=sr,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)

 mel_db = librosa.power_to_db(
    mel,
    ref=np.max
)

 return mel_db


In [ ]:
mel = audio_to_mel(
    df.iloc[0]["file"]
)

plt.figure(figsize=(12, 5))

librosa.display.specshow(
    mel,
    sr=16000,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar()

plt.title(
    "Cat sound: " + df.iloc[0]["label"]
)

plt.show()

In [ ]:
X = []
y = []
label_to_number = { "brushing": 0, "food": 1, "isolation": 2 }
for _, row in df.iterrows():

  try:

    mel = audio_to_mel(
        row["file"]
    )

    X.append(mel)

    y.append(
        label_to_number[row["label"]]
    )

  except Exception as e:

    print("Error:", e)
X = np.array(X)
y = np.array(y)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X = []
y = []
label_to_number = {
    "brushing": 0,
    "food": 1,
    "isolation": 2
}

for _, row in df.iterrows():
    try:
        mel = audio_to_mel(row["file"])
        X.append(mel)
        y.append(label_to_number[row["label"]])
    except Exception as e:
        print("Error:", e)

X = np.array(X)
y = np.array(y)

# Conv2D ожидает: (примеры, mel-частоты, кадры-времени, каналы).
# Канал добавляется ровно один раз.
if X.ndim == 3:
    X = X[..., np.newaxis]

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

In [ ]:
# Проверка перед train_test_split. Эту ячейку можно запускать повторно.
if X.ndim != 4 or X.shape[-1] != 1:
    raise ValueError(f"Ожидалась форма (N, 128, time, 1), получено: {X.shape}")

print("CNN input shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Testing:", len(X_test))

In [ ]:
model = Sequential([
 Conv2D(
    32,
    (3, 3),
    activation="relu",
    padding="same",
    input_shape=X_train.shape[1:]
),

 MaxPooling2D((2, 2)),

 Conv2D(
    64,
    (3, 3),
    activation="relu",
    padding="same"
),

 MaxPooling2D((2, 2)),

 Conv2D(
    128,
    (3, 3),
    activation="relu",
    padding="same"
),

 MaxPooling2D((2, 2)),

 Dropout(0.3),

 Flatten(),

 Dense(
    128,
    activation="relu"
),

 Dropout(0.4),

 Dense(
    3,
    activation="softmax"
)
])
model.summary()


In [ ]:
# Модель уже создана в предыдущей ячейке.
# Не создаём её повторно, чтобы не потерять обученные веса при повторном запуске.
model.summary()


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    X_train,
    y_train,
    epochs=40,
    batch_size=16,
    validation_split=0.2
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(
    f"Test accuracy: {accuracy * 100:.2f}%"
)

In [ ]:
predictions = model.predict(X_test)

y_pred = np.argmax(
    predictions,
    axis=1
)

class_names = [
    "brushing",
    "food",
    "isolation"
]

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names
    )
)

In [ ]:
model.save(
    "/content/cat_translator.keras"
)

print("🐈 Model saved!")

In [ ]:
my_file = choose_file("Введите путь к WAV-файлу кошки: " )

In [ ]:
mel = audio_to_mel(
    my_file
)

mel = mel[np.newaxis, ..., np.newaxis]

prediction = model.predict(
    mel,
    verbose=0
)[0]

In [ ]:
for name, probability in zip( class_names, prediction ):

   print(
    f"{name}: {probability * 100:.2f}%"
)
best_index = np.argmax(prediction)
print("\n🐈 CAT TRANSLATOR")
print("----------------------")
print( "Most likely:", class_names[best_index] )
print( "Confidence:", f"{prediction[best_index] * 100:.2f}%" )